In [0]:
import requests
import json

context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
token = context.apiToken().get()
base_url = context.apiUrl().get()

# Read a Bronze table profile
df = spark.read.table("globalmart.bronze.customers")
skip_cols = {"source_file", "load_timestamp", "_source_file", "_load_timestamp", "rescued_data"}
columns = []
for c in df.columns:
    if c.lower() in skip_cols:
        continue
    samples = [str(row[0]) for row in df.select(c).filter(df[c].isNotNull()).limit(5).collect()]
    columns.append({"name": c, "type": str(df.schema[c].dataType), "samples": samples})

print(f"Bronze columns found: {[c['name'] for c in columns]}")

# Call LLM
prompt = f"""You are a data engineering assistant. Map source columns to canonical Silver columns.

BRONZE COLUMNS:
{json.dumps(columns, indent=2)}

TARGET SILVER COLUMNS:
customer_id (STRING, primary key, format XX-NNNNN)
customer_email (STRING, optional)
customer_name (STRING, required)
segment (STRING, must be Consumer/Corporate/Home Office, may have abbreviations CONS/CORP/HO or typo Cosumer)
country, city, state, postal_code, region (all STRING, optional)

RESPOND WITH ONLY a JSON array. No markdown, no backticks:
[{{"source_column": "...", "canonical_column": "...", "transformation": null or "SQL expression using {{col}}"}}]"""

response = requests.post(
    f"{base_url}/serving-endpoints/databricks-gpt-oss-20b/invocations",
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={"messages": [{"role": "user", "content": prompt}], "temperature": 0.1, "max_tokens": 4000},
    timeout=120
)

print(f"\nStatus: {response.status_code}")
content = response.json()["choices"][0]["message"]["content"]

# Extract text block
if isinstance(content, list):
    for block in content:
        if isinstance(block, dict) and block.get("type") == "text":
            raw_text = block["text"].strip()
            break
else:
    raw_text = content.strip()

print(f"\nRaw LLM output:\n{raw_text}")

try:
    if raw_text.startswith("```"):
        raw_text = raw_text.split("\n", 1)[1]
    if raw_text.endswith("```"):
        raw_text = raw_text.rsplit("\n", 1)[0]
    mappings = json.loads(raw_text)
    print(f"\nParsed {len(mappings)} mappings:")
    for m in mappings:
        print(f"  {m['source_column']} -> {m['canonical_column']}" + (f" | {m['transformation']}" if m.get('transformation') else ""))
except Exception as e:
    print(f"\nJSON parse failed: {e}")

In [0]:
import sys
from pyspark.sql.functions import (
    col, coalesce, expr, lit, current_timestamp, when,
    monotonically_increasing_id
)
sys.path.append("/Workspace/dbx_retail/Globalmart")

import json
import mlflow.deployments
def call_llm_for_mapping(entity: str, bronze_profile: dict, target_schema: dict) -> list:
    """
    Sends Bronze profile + target schema to LLM and gets back column mappings
    with transformation expressions.

    Returns list of dicts matching column_mapping format:
    [{"source_column": "...", "canonical_column": "...", "transformation": "..."}, ...]
    """
    client = mlflow.deployments.get_deploy_client("databricks")

    prompt = f"""You are a data engineering assistant. Your job is to map source columns from a Bronze table to canonical Silver columns.

BRONZE TABLE: {entity}
BRONZE COLUMNS (with sample values):
{json.dumps(bronze_profile["columns"], indent=2)}

TARGET SILVER SCHEMA:
{json.dumps(target_schema["columns"], indent=2)}

RULES:
1. For each Bronze column, determine which canonical Silver column it maps to based on column name similarity AND sample data patterns.
2. Multiple Bronze columns may map to the same canonical column (they will be COALESCEd).
3. If a column needs transformation to match the target type, provide a SQL expression using {{col}} as a placeholder for the source column name.
4. Transformation examples:
   - Type casting: "CAST({{col}} AS INT)"
   - Dollar sign removal + cast: "CAST(REGEXP_REPLACE({{col}}, '^\\\\$', '') AS DECIMAL(10,2))"
   - Dual date format parsing: "COALESCE(TRY_TO_TIMESTAMP({{col}}, 'MM/dd/yyyy HH:mm'), TRY_TO_TIMESTAMP({{col}}, 'yyyy-MM-dd HH:mm'))"
   - ISO timestamp: "TRY_TO_TIMESTAMP({{col}})"
   - Date only: "TO_DATE({{col}}, 'yyyy-MM-dd')"
   - Placeholder replacement: "CASE WHEN {{col}} = '?' THEN NULL ELSE {{col}} END"
   - Abbreviation mapping: "CASE WHEN {{col}} IN ('CONS', 'Cosumer') THEN 'Consumer' WHEN {{col}} = 'CORP' THEN 'Corporate' WHEN {{col}} = 'HO' THEN 'Home Office' ELSE {{col}} END"
5. If a Bronze column does not match any canonical column, skip it.
6. If a canonical column has no matching Bronze column, skip it (it will be NULL).

RESPOND WITH ONLY a JSON array. No explanation, no markdown, no backticks. Example:
[
  {{"source_column": "cust_id", "canonical_column": "customer_id", "transformation": null}},
  {{"source_column": "order_date", "canonical_column": "order_purchase_timestamp", "transformation": "COALESCE(TRY_TO_TIMESTAMP({{col}}, 'MM/dd/yyyy HH:mm'), TRY_TO_TIMESTAMP({{col}}, 'yyyy-MM-dd HH:mm'))"}}
]"""

    response = client.predict(
        endpoint="databricks-claude-sonnet-4-20250514",
        inputs={
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 4000,
            "temperature": 0.0
        }
    )

    raw_text = response.choices[0]["message"]["content"].strip()

    # Clean any accidental markdown wrapping
    if raw_text.startswith("```"):
        raw_text = raw_text.split("\n", 1)[1]
    if raw_text.endswith("```"):
        raw_text = raw_text.rsplit("\n", 1)[0]

    mappings = json.loads(raw_text)
    return mappings
# Copy the get_bronze_profile and TARGET_SCHEMAS from silver_llm.py
# Then test one entity:
TARGET_SCHEMAS = {
    "customers": {
        "columns": {
            "customer_id": {"type": "STRING", "role": "primary_key", "description": "Unique customer identifier, format XX-NNNNN"},
            "customer_email": {"type": "STRING", "role": "optional", "description": "Customer email address"},
            "customer_name": {"type": "STRING", "role": "required", "description": "Full name of the customer"},
            "segment": {"type": "STRING", "role": "required", "description": "Business segment: must be one of Consumer, Corporate, Home Office. May appear as abbreviations (CONS, CORP, HO) or typos (Cosumer)"},
            "country": {"type": "STRING", "role": "optional"},
            "city": {"type": "STRING", "role": "optional"},
            "state": {"type": "STRING", "role": "optional"},
            "postal_code": {"type": "STRING", "role": "optional"},
            "region": {"type": "STRING", "role": "optional"}
        }
    },
    "orders": {
        "columns": {
            "order_id": {"type": "STRING", "role": "primary_key"},
            "customer_id": {"type": "STRING", "role": "foreign_key"},
            "vendor_id": {"type": "STRING", "role": "foreign_key"},
            "ship_mode": {"type": "STRING", "role": "optional"},
            "order_status": {"type": "STRING", "role": "required"},
            "order_purchase_timestamp": {"type": "TIMESTAMP", "role": "required", "description": "May be in MM/dd/yyyy HH:mm or yyyy-MM-dd HH:mm format"},
            "order_approved_timestamp": {"type": "TIMESTAMP", "role": "optional", "description": "Same dual format as purchase date"},
            "order_delivered_carrier_timestamp": {"type": "TIMESTAMP", "role": "optional"},
            "order_delivered_customer_timestamp": {"type": "TIMESTAMP", "role": "optional"},
            "order_estimated_delivery_timestamp": {"type": "TIMESTAMP", "role": "optional"}
        }
    },
    "transactions": {
        "columns": {
            "order_id": {"type": "STRING", "role": "foreign_key"},
            "product_id": {"type": "STRING", "role": "foreign_key"},
            "sales": {"type": "DECIMAL(10,2)", "role": "required", "description": "Sale amount. May have $ prefix that needs stripping"},
            "quantity": {"type": "INT", "role": "required"},
            "discount": {"type": "DECIMAL(5,2)", "role": "optional"},
            "profit": {"type": "DECIMAL(10,2)", "role": "optional", "description": "May be missing entirely from some sources"},
            "payment_type": {"type": "STRING", "role": "optional", "description": "May contain ? placeholder meaning unknown"},
            "payment_installments": {"type": "INT", "role": "optional"}
        }
    },
    "returns": {
        "columns": {
            "order_id": {"type": "STRING", "role": "foreign_key", "description": "Links to orders table"},
            "refund_amount": {"type": "DECIMAL(10,2)", "role": "required", "description": "May have $ prefix"},
            "return_date": {"type": "DATE", "role": "required", "description": "Format yyyy-MM-dd"},
            "return_reason": {"type": "STRING", "role": "optional", "description": "May contain ? placeholder meaning unknown"},
            "return_status": {"type": "STRING", "role": "required", "description": "Must be Pending, Approved, or Rejected. May appear as abbreviations: PENDG, APPRVD, RJCTD"}
        }
    },
    "products": {
        "columns": {
            "product_id": {"type": "STRING", "role": "primary_key"},
            "product_name": {"type": "STRING", "role": "required"},
            "brand": {"type": "STRING", "role": "optional"},
            "categories": {"type": "STRING", "role": "optional"},
            "colors": {"type": "STRING", "role": "optional"},
            "manufacturer": {"type": "STRING", "role": "optional"},
            "dimension": {"type": "STRING", "role": "optional"},
            "sizes": {"type": "STRING", "role": "optional"},
            "upc": {"type": "STRING", "role": "optional"},
            "weight": {"type": "STRING", "role": "optional"},
            "product_photos_qty": {"type": "BIGINT", "role": "optional"},
            "date_added": {"type": "TIMESTAMP", "role": "optional", "description": "ISO 8601 format"},
            "date_updated": {"type": "TIMESTAMP", "role": "optional", "description": "ISO 8601 format"}
        }
    },
    "vendors": {
        "columns": {
            "vendor_id": {"type": "STRING", "role": "primary_key"},
            "vendor_name": {"type": "STRING", "role": "required"}
        }
    }
}

# COMMAND ----------

# =============================================================================
# LLM SCHEMA MAPPER — Calls Foundation Model to generate mappings
# =============================================================================

def get_bronze_profile(entity: str) -> dict:
    """
    Reads a Bronze table and returns column names, types, and sample values
    for the LLM to analyze.
    """
    df = spark.read.table(f"globalmart.bronze.{entity}")

    profile = {"entity": entity, "columns": []}
    # Exclude metadata columns
    skip_cols = {"source_file", "load_timestamp", "_source_file", "_load_timestamp", "rescued_data"}

    for c in df.columns:
        if c.lower() in skip_cols:
            continue
        col_type = str(df.schema[c].dataType)
        # Get up to 5 non-null sample values
        samples = [
            str(row[0]) for row in
            df.select(c).filter(col(c).isNotNull()).limit(5).collect()
        ]
        profile["columns"].append({
            "name": c,
            "type": col_type,
            "samples": samples
        })

    return profile


entity = "customers"
profile = get_bronze_profile(entity)
target = TARGET_SCHEMAS[entity]

print("=== BRONZE PROFILE SENT TO LLM ===")
print(json.dumps(profile, indent=2))

print("\n=== CALLING LLM ===")
try:
    mappings = call_llm_for_mapping(entity, profile, target)
    print("\n=== LLM RESPONSE ===")
    print(json.dumps(mappings, indent=2))
except Exception as e:
    print(f"\n=== LLM FAILED ===")
    print(f"Error: {str(e)}")
    print(f"Type: {type(e).__name__}")

In [0]:
from openai import OpenAI

context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
client = OpenAI(
    api_key=context.apiToken().get(),
    base_url=f"{context.apiUrl().get()}/serving-endpoints"
)

response = client.chat.completions.create(
    model="databricks-gpt-oss-20b",
    messages=[{"role": "user", "content": "Return a JSON array: [{\"test\": true}]"}],
    temperature=0.1,
    max_tokens=100
)

print(response.choices[0].message.content)

In [0]:
%sql
-- Which method was used for customers?
SELECT _mapping_source, COUNT(*) 
FROM globalmart.silver.customers 
GROUP BY _mapping_source;



In [0]:
%sql
-- Were any entities mapped via fallback?
SELECT 'customers' AS entity, _mapping_source FROM globalmart.silver.customers
UNION ALL
SELECT 'orders', _mapping_source FROM globalmart.silver.orders 
UNION ALL
SELECT 'transactions', _mapping_source FROM globalmart.silver.transactions
UNION ALL
SELECT 'returns', _mapping_source FROM globalmart.silver.returns 
UNION ALL
SELECT 'products', _mapping_source FROM globalmart.silver.products 
UNION ALL
SELECT 'vendors', _mapping_source FROM globalmart.silver.vendors

In [0]:
%sql
select * from globalmart.bronze.transactions where rescued_data is not null

In [0]:
dbutils.fs.rm("/Volumes/globalmart/bronze/pipeline_metadata/schemas", recurse=True)

In [0]:
%sql
select count(*) as c, customer_id, region, source_file  from globalmart.bronze.customers
where source_file like '%6%'
group by customer_id, region, source_file having count(*) > 1


In [0]:
%sql
-- 1. Check what columns actually exist in each bronze table after Auto Loader merge
DESCRIBE TABLE globalmart.bronze.customers;
DESCRIBE TABLE globalmart.bronze.orders;
DESCRIBE TABLE globalmart.bronze.transactions;
DESCRIBE TABLE globalmart.bronze.returns;
DESCRIBE TABLE globalmart.bronze.products;
DESCRIBE TABLE globalmart.bronze.vendors;

In [0]:
%sql
-- Check if harmonized customers have any nulls in customer_email
SELECT 
    COUNT(*) AS total,
    SUM(CASE WHEN customer_email IS NULL THEN 1 ELSE 0 END) AS null_email,
    SUM(CASE WHEN segment NOT IN ('Consumer', 'Corporate', 'Home Office') THEN 1 ELSE 0 END) AS bad_segment
FROM globalmart.silver.customers;

In [0]:
%sql
-- Check what columns exist in the quarantine tables
DESCRIBE TABLE globalmart.silver.customers_quarantine;

In [0]:
import sys
sys.path.append("/Workspace/dbx_retail/Globalmart")

from utilities.utils import harmonize_columns, load_dq_rules

bronze = spark.read.table("globalmart.bronze.customers")
harmonized = harmonize_columns(bronze, spark, "customers")

print("Harmonized columns:", harmonized.columns)
print("Total rows:", harmonized.count())
print("Null emails:", harmonized.filter("customer_email IS NULL").count())

rules = load_dq_rules(spark, "customers")
for r in rules:
    print(f"Rule: {r['rule_name']} | Expression: {r['rule_expression']}")
    try:
        fail_count = harmonized.filter(f"NOT ({r['rule_expression']})").count()
        print(f"  → Failures: {fail_count}")
    except Exception as e:
        print(f"  → ERROR: {e}")